In [1]:
import pyodbc
import pandas as pd

In [21]:
df = pd.read_parquet('combined_file_V2.parquet')

In [ ]:
supplied_energy = consumed_energy + distribution_loss (5-10%)

In [25]:
df.head(60)

,PlantId,IST_date,prediction_30,prediction_60,prediction_365,file_creation_date
0,1,2024-11-10 00:00:00,0.448996,0.637259,0.289048,2024-11-10
1,1,2024-11-10 01:00:00,0.448806,0.600679,0.289048,2024-11-10
2,1,2024-11-10 02:00:00,0.448806,0.600679,0.289048,2024-11-10
3,1,2024-11-10 03:00:00,0.448806,0.600679,0.289048,2024-11-10
4,1,2024-11-10 04:00:00,0.945617,0.860613,0.378399,2024-11-10
5,1,2024-11-10 05:00:00,1.076759,0.803585,0.430299,2024-11-10
6,1,2024-11-10 06:00:00,1.686839,1.163289,0.595355,2024-11-10
7,1,2024-11-10 07:00:00,2.215267,1.784479,1.026279,2024-11-10
8,1,2024-11-10 08:00:00,2.070791,2.610781,1.785976,2024-11-10
9,1,2024-11-10 09:00:00,5.400448,5.081501,2.825439,2024-11-10


In [3]:
df = pd.read_csv('Jan_2025.csv')

In [9]:
df[df['PlantCode']=='HINUP-01-06'].sort_values('CreatedonLocalTime')[60*10-20:].head(60)

,CreatedonLocalTime,PlantCode,ConnectionDeviceId,BatterySOC,SupplyPower,SolarPower,BiomassPower
189601,2025-01-07 06:30:00,HINUP-01-06,Husk-IOT-B00005,29.330000,2.020000,0.00,0.000000
188135,2025-01-07 06:45:00,HINUP-01-06,Husk-IOT-B00005,29.160000,2.130000,0.00,0.000000
188136,2025-01-07 07:00:00,HINUP-01-06,Husk-IOT-B00005,28.990000,2.190000,0.00,0.000000
188137,2025-01-07 07:15:00,HINUP-01-06,Husk-IOT-B00005,28.820000,2.330000,0.00,0.000000
188138,2025-01-07 07:30:00,HINUP-01-06,Husk-IOT-B00005,28.650000,2.760000,0.00,0.000000
186904,2025-01-07 07:45:00,HINUP-01-06,Husk-IOT-B00005,28.400000,3.160000,0.09,0.000000
189602,2025-01-07 08:00:00,HINUP-01-06,Husk-IOT-B00005,28.200001,2.710000,0.17,0.000000
186905,2025-01-07 08:15:00,HINUP-01-06,Husk-IOT-B00005,27.850000,1.620000,0.18,0.000000
188169,2025-01-07 08:30:00,HINUP-01-06,Husk-IOT-B00005,27.639999,1.650000,0.21,0.000000
188170,2025-01-07 08:45:00,HINUP-01-06,Husk-IOT-B00005,27.559999,1.910000,0.41,0.000000


In [3]:
server = "huskpowersqlserver.database.windows.net"
database = "HuskNeuronsSQLDB"
username='kshitizk'
password='Husk@17July2024K'
connection_string = pyodbc.connect('DRIVER={ODBC Driver 17 for SQL Server};SERVER='+server+';DATABASE='+database+';UID='+username+';PWD='+ password)

query = f"""
select CreatedonLocalTime, PlantCode, ConnectionDeviceId, BatterySOC,
        PlantlifetimeEnergy, MPPTS1Energy, MPPTS2Energy, MPPTS3Energy,BiomasslifetimeEnergy,
       SupplyPower, SolarPower, BiomassPower, BatteryCurrent,
       BatteryDCVoltage, BatteryChargeAh, BatteryDischargeAh
from OperationDashboardLive
where CreatedonLocalTime >= '2025-06-01' and CreatedonLocalTime<'2025-07-01' and PlantCode = 'HINUP-02-02'
"""

df = pd.read_sql(query, connection_string)
connection_string.close()

C:\Users\hps10\AppData\Local\Temp\ipykernel_39724\660273106.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, connection_string)


In [4]:
df.to_csv('004/Jun_2025_004.csv',index=False)

In [1]:
def old_battery_cost_per_kWh_function():
    energy_given_by_battery_in_one_cycle = bat_cap*dod/100
    energy_given_by_battery = energy_given_by_battery_in_one_cycle*give_cycles_from_dod(dod)*0.75 #derating of cycles
    total_battery_cost = cost_per_kWh*bat_cap
    cost_per_kWh = (total_battery_cost/energy_given_by_battery)*battery_cost_scaling_factor
    return cost_per_kWh

In [ ]:
def simulation_reward_function(new_soc, dg_power, current_reward, current_reward_weight_in_percent,
                               i, start_time, load, gp, hours=24):
    end_time = start_time+timedelta(hours=hours)
    start_time+=timedelta(minutes=15)
    current_time = start_time
    i+=1
    
    weights_sum = (hours*(hours+1))/2
    current_reward_weight = current_reward_weight_in_percent/100
    reward = -(current_reward_weight*weights_sum)/(1-current_reward_weight)
    
    while start_time<=end_time:
        t = start_time
        y,m,wd,d,h,mi = t.year,t.month,t.weekday(),t.day,t.hour,t.minute
        hour_difference = (t-current_time).total_seconds() / 3600
        increment = 1+(hour_difference//6)
        # print(t,h,t.date())
        GP = gp[(gp['local_hour']==h) & (gp['local_date']==pd.to_datetime(t.date()))]['solar_forecast'].iloc[0]
        state = [y,m,wd,d,h,mi,
                  new_soc,
                  load['load_forecast'].iloc[i],
                  GP,
                  dg_power]    
        action = policy(state)        
        new_soc, avg_soc, batttery_discharge_power, dg_power, solar_power_wasted, load_power_left = environment(state,action.numpy())
        tc,bc,dc,sc,lc = reward_function(avg_soc, battery_discharge_power,
                                         dg_power,dg_fuel_cost_per_kWh,
                                         solar_power_wasted,solar_cost_per_kWh,
                                         load_power_left, load_cost_per_kWh, minutes = 15*increment)
        
        reward += ((hours-hour_difference)/weights_sum)*tc
        i += increment
        i = int(i)
        start_time += timedelta(minutes=15*increment)
    return reward

In [ ]:
    #for soc less than 17, the DG will automatically start below 235V
    # if soc>=0 and soc<17:
    #     bcp = max(20.74,20.74)
    # elif soc>=17 and soc<30:
    #     bcp = max(15.86,15.86)
    # elif soc>=30 and soc<47:
    #     bcp = min(20.74,10.98)
    # elif soc>=47 and soc<60:
    #     bcp = min(15.86,10.98)
    # elif soc>=60 and soc<73:
    #     bcp = min(15.86,1)
    # elif soc>=73 and soc<82:
    #     bcp = min(10.98,1)
    # elif soc>=82 and soc<88:
    #     bcp = min(7.32,1)
    # elif soc>=88 and soc<92:
    #     bcp = min(4.88,1)
    # elif soc>=92 and soc<95:
    #     bcp = min(3.66,1)
    # elif soc>=95 and soc<97:
    #     bcp = min(2.44,1)
    # elif soc>=97 and soc<100:
    #     return 1
    # else:
    #     return 0

In [ ]:
bAh = pd.read_csv('BatteryAhData.csv')
bAh['ConnectionDeviceId']=''
for i in bAh.index:
    n = len(str(bAh['IOT'][i]))
    zeroes = 5-n
    bAh['ConnectionDeviceId'][i] = "Husk-IOT-B"+"0"*zeroes+str(bAh['IOT'][i])

bAh = bAh.drop(['IOT','Ah','CellInSeries'],axis=1)

In [ ]:
pcs1 = df['PlantCode'].unique()
pcs2 = solar['PlantCode'].unique()

In [ ]:
# solar.drop(['local_date','local_hour','update','dense_rank'],axis=1).groupby('PlantCode').mean().sort_values('pv_power_rooftop')
# #gp is generation potential
gp = solar[(solar['PlantCode']=='HINUP-02-02')]

# load = df[df['PlantCode']=='HINUP-01-14']
# load['CreatedonLocalTime'] = pd.to_datetime(load['CreatedonLocalTime'])
# load = load[['CreatedonLocalTime','PlantCode','BatterySOC','SolarPower','SupplyPower','BiomassPower']]
# load['minute'] = ''
# for i in load.index:
#     load['minute'][i] = load['CreatedonLocalTime'][i].minute
# load = load[load['minute'] %15==0]

## Creating Load Forecast

In [ ]:
l = [100+random.randint(-20, 20) for _ in range(len(load))]
cnt=0
load['load_forecast'] = 0

for i in load.index:
    load['load_forecast'][i] = (l[cnt]/100)*load['SupplyPower'][i]
    cnt+=1

load['month'] = ''
load['day'] = ''
load['hour'] = ''

for i in load.index:
    load['month'][i] = load['CreatedonLocalTime'][i].month
    load['day'][i] = load['CreatedonLocalTime'][i].day
    load['hour'][i] = load['CreatedonLocalTime'][i].hour

load['load_forecast'] = load[['load_forecast','month','day','hour']].groupby(['month','day','hour']).transform('mean')

## Creating Solar Forecast

In [ ]:
s = [100+random.randint(-20, 20) for _ in range(len(gp))]
cnt=0
gp['solar_forecast']=0
for i in gp.index:
    gp['solar_forecast'][i] = (l[cnt]/100)*gp['pv_power_rooftop'][i]
    cnt+=1

In [ ]:
x = pd.DataFrame(m2).drop(['episode','pc','actual_dg_power','avg_soc'],axis=1)#.to_excel('Partawal_Chowk_Apr_25_204.xlsx',index=False)

x['date'] = ''
for i in x.index:
 x['date'][i] = x['time'][i].date()
y = x.drop(['time'],axis=1).groupby('date').sum()/4
# y.to_excel('Fazilnagar_daily_Apr_2025_204.xlsx')

In [ ]:
score = -sc-sp+min(0,dgp+dc)+dtc
x,y = 0,0
# score = 100*((sc-dtc)/max(sp,1))*(-1)
if score>=85:
    # print("Jackpot Reward!!")
    x = 1
if soc<=38:
    y = 1
if score-prev_score<0 or score<0:
    return score-y*200-y*(38-soc)*50,round(score-prev_score,1)/np.sqrt(abs(score if score!=0 else 1))
else:
    return score-y*200-y*(38-soc)*50,round(score-prev_score,1)*np.sqrt(abs(score))